# 1. Introduction

This notebook provides an interactive walkthrough for exploring a single stock's fundamentals and valuation history. You can select a ticker, fetch the latest fundamentals, and review how valuation metrics evolve over time.

The workflow loads fundamentals and daily price data, summarizes key metrics in a concise snapshot, and visualizes the price-to-earnings (P/E) trend alongside a rolling average. Additional price charts help you quickly gauge the recent trend before drawing conclusions.


## 2. Imports

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from stock_project.src.analysis.fundamentals import summarize_fundamentals
from stock_project.src.analysis.valuation import build_pe_series, compute_rolling_pe
from stock_project.src.data.prices import get_price_history
from stock_project.src.viz.plots import plot_pe_with_rolling


## 3. User Input

Type a ticker, pick a lookback period, and choose a rolling window for smoothing valuation metrics. Click **Run analysis** to fetch data and render the visuals.

In [ ]:

# Interactive controls

ticker_input = widgets.Text(
    value="AAPL",
    placeholder="Enter ticker (e.g., AAPL)",
    description="Ticker:",
    style={"description_width": "initial"},
    disabled=False,
)

period_dropdown = widgets.Dropdown(
    options=["1y", "3y", "5y"],
    value="5y",
    description="Period:",
    style={"description_width": "initial"},
)

rolling_slider = widgets.IntSlider(
    value=30,
    min=7,
    max=365,
    step=7,
    description="Rolling window (days):",
    continuous_update=False,
    style={"description_width": "initial"},
)

run_button = widgets.Button(
    description="Run analysis",
    button_style="primary",
    tooltip="Fetch data and update charts",
    icon="line-chart",
)

controls = widgets.VBox([
    widgets.HBox([ticker_input, period_dropdown]),
    rolling_slider,
    run_button,
])

output = widgets.Output()

display(controls)
display(output)


## 4. Load Raw Data

Data loading happens inside the interactive callback. It fetches fundamentals, price history, and valuation series based on the selected ticker and period.

## 5. Fundamentals Summary

A concise snapshot of fundamental metrics is displayed to give a quick overview of the company's recent performance.

## 6. P/E History

The notebook builds a P/E series and overlays a configurable rolling average to highlight valuation trends over time.

## 7. Additional Plots (Optional)

A simple price chart (with a 30-day moving average) provides extra context about the stock's recent trading trend.

In [ ]:


def run_analysis(_=None):
    ticker = ticker_input.value.strip().upper()
    period = period_dropdown.value
    rolling_window = int(rolling_slider.value)

    with output:
        clear_output()
        if not ticker:
            print("Please enter a valid ticker symbol.")
            return

        print(f"Running analysis for {ticker} | Period: {period} | Rolling: {rolling_window} days")

        try:
            fundamentals = summarize_fundamentals(ticker)
            price_history = get_price_history(ticker, period=period, interval="1d")
            pe_series = build_pe_series(ticker, period=period)
            rolling_pe = compute_rolling_pe(pe_series, window=rolling_window)
        except Exception as exc:  # noqa: BLE001
            print("⚠️ Unable to load data. Please confirm the ticker and try again.")
            print(f"Details: {exc}")
            return

        # Fundamentals snapshot
        print(f"
### Fundamental snapshot for {ticker}")
        if fundamentals is None:
            print("No fundamentals returned.")
        else:
            if isinstance(fundamentals, dict):
                fundamentals_data = fundamentals
            elif hasattr(fundamentals, "model_dump"):
                fundamentals_data = fundamentals.model_dump()
            else:
                fundamentals_data = vars(fundamentals)

            fundamentals_df = pd.DataFrame([fundamentals_data])
            display(fundamentals_df)

        # P/E plot
        if pe_series is None or pe_series.empty:
            print("No P/E data available for plotting.")
        else:
            print("
P/E history with rolling average")
            fig_pe = plot_pe_with_rolling(pe_series, rolling_pe)
            display(fig_pe)
            plt.close(fig_pe)

        # Additional price plot
        if price_history is None or price_history.empty:
            print("No price history available.")
            return

        price_chart = price_history.copy()
        if "Close" in price_chart.columns:
            price_chart["Close_MA30"] = price_chart["Close"].rolling(window=30, min_periods=1).mean()

        fig, ax = plt.subplots(figsize=(10, 4))
        price_chart["Close"].plot(ax=ax, label="Close", color="steelblue")
        if "Close_MA30" in price_chart:
            price_chart["Close_MA30"].plot(ax=ax, label="30-day MA", color="orange", linestyle="--")
        ax.set_title(f"Price history for {ticker} ({period})")
        ax.set_xlabel("Date")
        ax.set_ylabel("Price")
        ax.legend()
        ax.grid(True, linestyle=":", alpha=0.7)
        plt.tight_layout()
        display(fig)
        plt.close(fig)


run_button.on_click(run_analysis)


## 8. Conclusions

Use the fundamentals snapshot to understand the company's recent financial position, then compare the latest P/E to its rolling history to gauge valuation trends. The price chart offers additional context about momentum or reversals, helping you decide whether deeper research is warranted.